# Entity ***Tokens***
+ Layer **gold**
+ Priority **1**
---
### Imports

In [0]:
%run ../common/medallion_functions

### Parameters

In [0]:
layer = dbutils.widgets.get("layer")
entity_name = dbutils.widgets.get("entity_name")
load_pattern = dbutils.widgets.get("load_pattern")

### Execution

In [0]:
sv_tokens_df = spark.read.table(f"uniswap.silver.tokens")

In [0]:
gld_tokens_df = spark.sql(f"""
    WITH cte_silver_tokens AS(
        SELECT
            pk_token_id AS pk_token_contract,
            symbol,
            name,
            decimals,
            v3_held_total_supply,
            derived_price_ETH,
            fees_USD,
            total_value_locked_USD,
            total_value_locked_token,
            volume_USD,
            volume_token,
            tx_count,
            pool_count
        FROM {{df}}
    )
    SELECT
        tokens.*,
        CASE
            WHEN tokens.symbol IN (
                "USDT", "USDT0",
                "USDC", "EURC",
                "USDS", "DAI",
                "USD1",
                "USDE",
                "USDH",
                "PYUSD",
                "RLUSD",
                "GHO",
                "USDAI",
                "CRVUSD",
                "FRXUSD",
                "AUSD",
                "MUSD",
                "USDG",
                "USDY",
                "PUSD",
                "TUSD",
                "USDF",
                "FDUSD",
                "USDD",
                "USDXL",
                "USDHL",
                "USDTB",
                "BOLD"
            ) THEN "stablecoin"
            WHEN tokens.symbol IN (
                "WBTC",
                "WETH",
                "CBBTC",
                "CBETH",
                "BBTC",
                "LBTC",
                "SBTC",
                "WSTETH",
                "STETH"
            ) THEN "wrapped_asset"
            ELSE "other"
        END AS token_type,
        CASE
            WHEN tokens.total_value_locked_USD < 10000 THEN TRUE
            ELSE FALSE
        END AS is_low_tvl_token,
        context.network,
        context.protocol_version,
        CURRENT_TIMESTAMP() AS _load_timestamp_gold
    FROM cte_silver_tokens AS tokens
    CROSS JOIN uniswap.silver.factory AS context
""", df = sv_tokens_df)

### Save & exit

In [0]:
load_result = load_entity(gld_tokens_df,
                        entity_name,
                        load_pattern,
                        layer
                        )

In [0]:
dbutils.notebook.exit(load_result)